In [9]:
import os
import pandas as pd
import numpy as np
from numba import njit, float64, int64, uint64,types
from numba.typed import Dict

In [10]:

# 현재 파일들이 있는 그 위치 그대로 설정
# Base_dir = "C:/Users/user/Desktop/IDS_masters/9) Car-Hacking Dataset"
Base_dir = "C:/Users/user/Desktop/IDS_masters/audi/audi"

# 파일 이름에 포함된 단어로 공격 유형 구분
attack_mapping = {
    "Dos": 1,
    "Fuzzing": 2,
    "Spoofing":4
}

attack_files = []

# 폴더 안을 바로 검사
for attack_name, attack_id in attack_mapping.items():
    attack_dir = os.path.join(Base_dir, attack_name)
    if not os.path.isdir(attack_dir):
        continue

    for fname in os.listdir(attack_dir):
        if fname.endswith(".csv"):
            attack_files.append({
                "path": os.path.join(attack_dir, fname),
                "attack_id": attack_id
            })

In [11]:
#################################
# 2. Visualization Mirgu Dataset
#################################
hash_cache = {}

# [ADD] payload 8바이트 리스트로 만드는 함수 (너가 쓰던 스타일)
def parse_payload(row):
    # row에는 b0~b7 컬럼이 있고, 이미 0패딩되어 있음
    return [int(row[f"b{i}"]) for i in range(8)]

def process_csv_file(path, attack_id):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",")
            if len(parts) < 4:
                continue

            ts_str, canid_raw, dlc_str = parts[0], parts[1], parts[2]
            label = parts[-1].strip()         # [MINOR] strip
            data_tokens = parts[3:-1]

            # dlc/ts 파싱
            try:
                ts = float(ts_str)
                dlc = int(dlc_str)
            except:
                continue

            # payload bytes: DLC 만큼만 읽고, 8바이트로 0 패딩
            payload = []
            for i in range(min(dlc, len(data_tokens), 8)):
                tok = data_tokens[i].strip()
                if tok == "" or tok.lower() == "nan":
                    payload.append(0)
                else:
                    try:
                        payload.append(int(tok, 16))
                    except:
                        payload.append(0)

            payload += [0] * (8 - len(payload))
            payload = payload[:8]

            rows.append([ts, canid_raw, dlc, *payload, label])

    df = pd.DataFrame(
        rows,
        columns=["timestamp", "CAN_ID", "DLC"] + [f"b{i}" for i in range(8)] + ["Label"]
    )

    # CAN_ID int 변환
    df["int_CAN_ID"] = df["CAN_ID"].apply(lambda x: int(str(x).strip(), 16)).astype(np.int64)


    # Payloads 컬럼 추가 
    df["Payloads"] = df.apply(parse_payload, axis=1).tolist()

    # 라벨링
    df["Labeling"] = df["Label"].map({"T": attack_id, "R": 0}).fillna(0).astype(int)

    df = df[["timestamp","int_CAN_ID","DLC","Payloads","Labeling"]]

    return df


In [12]:
import numpy as np
from numba import njit, float64, int64, uint64, types
from numba.typed import Dict


@njit
def popcount64(x):
    c = 0
    v = int64(x)
    while v:
        v &= v - int64(1)
        c += 1
    return c


@njit
def pack_payload_u64_dlc(row, dlc):
    v = uint64(0)
    d = dlc
    if d < 0:
        d = 0
    if d > 8:
        d = 8
    for i in range(d):
        v |= uint64(row[i]) << (i * 8)
    return v


@njit
def same_id_recent_k_surprise(i, can_ids, dlcs, payloads, k_hist, eps):
    """
    현재 프레임 i의 CAN ID와 같은 ID의 최근 K개 메시지로 byte histogram을 만들고,
    현재 payload가 그 분포에서 얼마나 드문지 1개 scalar로 반환.
    """
    target_cid = int64(can_ids[i])
    hist = np.zeros(256, dtype=np.float64)

    found_msgs = 0
    total_bytes = 0.0

    # 최근 K개 same-ID 메시지에서 histogram 생성
    for j in range(i, -1, -1):
        if int64(can_ids[j]) != target_cid:
            continue

        d = int64(dlcs[j])
        if d < 0:
            d = 0
        elif d > 8:
            d = 8

        row = payloads[j]
        for b in range(d):
            hist[row[b]] += 1.0
            total_bytes += 1.0

        found_msgs += 1
        if found_msgs >= k_hist:
            break

    # 데이터가 없으면 0 반환
    if total_bytes <= 0.0:
        return 0.0

    # 확률화
    for v in range(256):
        hist[v] /= total_bytes

    # 현재 payload의 평균 negative log prob
    dcur = int64(dlcs[i])
    if dcur < 0:
        dcur = 0
    elif dcur > 8:
        dcur = 8

    if dcur == 0:
        return 0.0

    row_cur = payloads[i]
    s = 0.0
    for b in range(dcur):
        p = hist[row_cur[b]]
        s += -np.log(p + eps)

    return s / float64(dcur)


@njit(fastmath=True)
def calculate_features(timestamps, can_ids, dlcs, payloads, k_hist=32):

    n = len(timestamps)

    # feature index
    # 0  is_cid0
    # 1  dlc/8
    # 2  log1p(rel_change)/5
    # 3  log1p(ent * rel_change)
    # 4  freq_local
    # 5  id_ent
    # 6  streak_ratio_fixed
    # 7  norm_by_win_mean
    # 8  freq_over_top1_log
    # 9  top1_share_fixed
    # 10 idx_gap_cv
    # 11 dominance_ratio
    # 12 same_id_recent_k_surprise

    features = np.zeros((n, 13), dtype=np.float64)

    last_time_map    = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)

    local_cnt_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    id_ham_ema    = Dict.empty(key_type=types.int64, value_type=types.float64)
    streak_map    = Dict.empty(key_type=types.int64, value_type=types.int64)

    last_pos_map = Dict.empty(key_type=types.int64, value_type=types.int64)
    gap_n_map    = Dict.empty(key_type=types.int64, value_type=types.int64)
    gap_mean_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    gap_M2_map   = Dict.empty(key_type=types.int64, value_type=types.float64)

    alpha_ham = 0.05
    eps = 1e-9
    W = 128.0

    top1_id = int64(-1)
    top2_id = int64(-1)
    top1_cnt = 0.0
    top2_cnt = 0.0

    for i in range(n):

        if (i % 128) == 0:
            local_cnt_map.clear()
            streak_map.clear()
            last_pos_map.clear()
            gap_n_map.clear()
            gap_mean_map.clear()
            gap_M2_map.clear()

            top1_id = int64(-1)
            top2_id = int64(-1)
            top1_cnt = 0.0
            top2_cnt = 0.0

        cid = int64(can_ids[i])
        dlc = int64(dlcs[i])
        row = payloads[i]
        pos = int64(i % 128)

        if cid in last_time_map:
            curr_iat = timestamps[i] - last_time_map[cid]
            if curr_iat < 0.0:
                curr_iat = 0.0
        last_time_map[cid] = timestamps[i]

        if dlc < 0:
            dlc = 0
        elif dlc > 8:
            dlc = 8

        cur_bytes = pack_payload_u64_dlc(row, dlc)

        h_dist = 0.0
        rel_change = 0.0
        streak = int64(0)

        if cid in last_payload_map:
            diff_bits = cur_bytes ^ last_payload_map[cid]
            h_dist = float64(popcount64(diff_bits))

            if diff_bits == uint64(0):
                streak = streak_map.get(cid, int64(0)) + int64(1)
            else:
                streak = int64(0)
            streak_map[cid] = streak

            if cid in id_ham_ema:
                avg_h = id_ham_ema[cid]
                rel_change = h_dist / (avg_h + 0.1)
                id_ham_ema[cid] = (1.0 - alpha_ham) * avg_h + alpha_ham * h_dist
            else:
                rel_change = 1.0
                id_ham_ema[cid] = h_dist
        else:
            streak_map[cid] = int64(0)

        last_payload_map[cid] = cur_bytes

        ent = 0.0
        if dlc > 0:
            p_counts = np.zeros(256, dtype=np.int64)
            for k in range(dlc):
                p_counts[row[k]] += 1
            for c in p_counts:
                if c > 0:
                    p = c / float64(dlc)
                    ent -= p * np.log(p)

        id_ent = 0.0
        if i >= 127:
            win_id_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
            for j in range(i - 127, i + 1):
                wid = int64(can_ids[j])
                win_id_counts[wid] = win_id_counts.get(wid, 0.0) + 1.0
            for k_id in win_id_counts:
                pk = win_id_counts[k_id] / W
                id_ent -= pk * np.log(pk + 1e-9)
            id_ent = id_ent / 4.85

        cnt = local_cnt_map.get(cid, 0.0) + 1.0
        local_cnt_map[cid] = cnt
        freq_local = cnt / W

        num_unique = float64(len(local_cnt_map))
        total_seen = float64(pos + 1)
        mean_count = total_seen / (num_unique + eps)
        norm_by_win_mean = cnt / (mean_count + eps)

        if cid == top1_id:
            top1_cnt = cnt
        elif cid == top2_id:
            top2_cnt = cnt

        if cnt > top1_cnt + 1e-12:
            if cid != top1_id:
                top2_id = top1_id
                top2_cnt = top1_cnt
                top1_id = cid
                top1_cnt = cnt
        elif cnt > top2_cnt + 1e-12 and cid != top1_id:
            top2_id = cid
            top2_cnt = cnt

        freq_over_top1_log = np.log((cnt + eps) / (top1_cnt + eps))
        top1_share_fixed = top1_cnt / W
        dominance_ratio = (top1_cnt - top2_cnt) / (top1_cnt + eps)
        streak_ratio_fixed = float64(streak) / W

        idx_gap_cv = 0.0
        if cid in last_pos_map:
            gap = pos - last_pos_map[cid]
            if gap <= 0:
                gap = int64(1)

            if cid in gap_n_map:
                gn = gap_n_map[cid] + int64(1)
                gmean = gap_mean_map[cid]
                gM2 = gap_M2_map[cid]

                x = float64(gap)
                delta = x - gmean
                gmean = gmean + delta / float64(gn)
                delta2 = x - gmean
                gM2 = gM2 + delta * delta2

                gap_n_map[cid] = gn
                gap_mean_map[cid] = gmean
                gap_M2_map[cid] = gM2
            else:
                gap_n_map[cid] = int64(1)
                gap_mean_map[cid] = float64(gap)
                gap_M2_map[cid] = 0.0

            gn = gap_n_map[cid]
            gmean = gap_mean_map[cid]
            gM2 = gap_M2_map[cid]
            if gn >= 2:
                var = gM2 / float64(gn - 1)
                if var < 0.0:
                    var = 0.0
                gstd = np.sqrt(var)
                idx_gap_cv = gstd / (gmean + eps)

        last_pos_map[cid] = pos

        # 새 13번째 feature
        surprise_score = same_id_recent_k_surprise(
            i, can_ids, dlcs, payloads, k_hist, eps
        )

        features[i, 0] = 1.0 if cid == 0 else 0.0
        features[i, 1] = float64(dlc) / 8.0
        features[i, 2] = np.log1p(rel_change) / 5.0
        features[i, 3] = np.log1p(ent * rel_change)
        features[i, 4] = freq_local
        features[i, 5] = id_ent
        features[i, 6] = streak_ratio_fixed
        features[i, 7] = norm_by_win_mean
        features[i, 8] = freq_over_top1_log
        features[i, 9] = top1_share_fixed
        features[i, 10] = idx_gap_cv
        features[i, 11] = dominance_ratio
        features[i, 12] = surprise_score

    return features

In [13]:
# ==========================================
# 4. Making Feature with Numba
# ==========================================


def Make_feature(path, attack_id):

    df = process_csv_file(path, attack_id)

    # ======== to numpy ========== #
    timestamps = df["timestamp"].to_numpy(np.float32)
    can_ids = df["int_CAN_ID"].to_numpy(np.int64)
    payloads = np.array(df["Payloads"].tolist(), dtype=np.uint8)
    dlcs = df["DLC"].to_numpy(np.int64)
    labels = df["Labeling"].to_numpy(np.int64)

    
    # ======== calculate feature ========== #
    feature9 = calculate_features(timestamps, can_ids,dlcs ,payloads)
    print(feature9.shape)

    return feature9, labels

In [14]:
# ==========================================
# 5. Slide Window and Label
# ==========================================
def Sliding_Window_and_Labeling(feature, label, win_size=128, stride=64):
    windows = []
    labels = []
    n = feature.shape[0]
    for start in range(0, n-win_size+1 , stride):
        end = start + win_size
        windows.append(feature[start:end])
        labels.append(label[start:end])


    return (
        np.stack(windows, axis=0).astype(np.float32),
        np.stack(labels, axis=0).astype(np.int64)
    )

In [15]:
# ==========================================
# 6. main
# ==========================================
all_x = []
all_y = []

for item in attack_files:
    feature9, labels = Make_feature(item["path"], item["attack_id"]) # 각 feature 추출
    windows, y = Sliding_Window_and_Labeling(feature9,labels) # 윈도우 만들기

    all_x.append(windows)
    all_y.append(y)

all_x_win = np.concatenate(all_x, axis=0)
all_y_win = np.concatenate(all_y, axis=0)

(355183, 13)
(91012, 13)
(243741, 13)


In [16]:
# ==========================================
# 7. Save
# ==========================================
import numpy as np
np.savez(
    "D:/IDS_masters/dataset/audi_0305_537.npz",
    X = all_x_win.astype(np.float32),
    y = all_y_win.astype(np.int64)
    )

print(f" Saved dataset")

print("X shape:", all_x_win.shape)
print("y shape:", all_y_win.shape)

 Saved dataset
X shape: (10776, 128, 13)
y shape: (10776, 128)
